# Data access

Access GI.cloud buffer data, online values, and file exchange with Python. Recorded measurements have their own notebook, `measurements.ipynb`.

**In the Analytics tab:** this is a read-only preview. Download the notebook and open it in JupyterLab to run the examples with your own data.

Run **Setup**, then the section you need. Only the cells within **Buffer data** depend on one another; other sections need Setup only. Optional cells start with `run_... = False`: configure the inputs and set the switch to `True` to run. Reset it to `False` before using Run All. Sphinx and Dash display saved outputs.

**Output examples** use shortened IDs and are not results from your target. Optional previews show the enabled operation's result; disabled cells print a skip message.

## Setup

Install `pygidata`, `pandas`, `matplotlib`, and `python-dotenv` in your Jupyter kernel. Set `GI_BASE_URL` and either `GI_TOKEN` or `GI_USER` / `GI_PASSWORD` in your environment, or in a private `.env` file in your notebook's working directory. The setup cell loads this file without replacing existing environment variables.

Use the raw API token without a `Bearer` prefix. Edit each example's inputs in its own cell. Never save credentials in notebook cells, outputs, or version control.

In [ ]:
# Enable autoreload for debugging the library in notebooks.

%load_ext autoreload
%autoreload 2

In [ ]:
import logging
import os
from importlib.metadata import version

import pandas as pd
from dotenv import load_dotenv, find_dotenv
from gi_data.dataclient import GIDataClient
from gi_data.mapping.models import VarSelector

# find_dotenv() searches upward from the current working dir
env_path = find_dotenv(usecwd=True)
print("Loaded .env from:", env_path or "NOT FOUND")
load_dotenv(env_path, override=False)

base_url = os.environ["GI_BASE_URL"]

auth = (
    {"access_token": os.environ["GI_TOKEN"]}
    if os.getenv("GI_TOKEN")
    else {"username": os.environ["GI_USER"], "password": os.environ["GI_PASSWORD"]}
)
client = GIDataClient(base_url, **auth)
print(f"pygidata {version('pygidata')}")

In [ ]:
# Set log level for the client outputs
# Note: Set log level to debug for detailed request/response info from the backend
client.set_log_level(logging.INFO)  # INFO, DEBUG, WARNING, ERROR

## Buffer data

Discover streams and copy the chosen ID into `source_id` below. Use IDs, not list positions.

In [ ]:
sources = client.list_buffer_sources()
pd.DataFrame([
    {"id": str(s.id), "name": s.name, "first_ms": s.first_ts, "last_ms": s.last_ts}
    for s in sources
])

**Example output**

| id | name | first_ms | last_ms |
| --- | --- | ---: | ---: |
| 0 | Demo stream | 1699999999000 | 1700000001000 |

One row per source. Choose an `id` from your actual output.

Set `source_id` here. Then choose a numeric variable from the table for the read cell's `variable_id`.

In [ ]:
source_id = "0"
source = {str(s.id): s for s in sources}[source_id]
variables = client.list_buffer_variables(source.id)
pd.DataFrame([
    {"id": str(v.id), "name": v.name, "unit": v.unit}
    for v in variables
])

**Example output**

| id | name | unit |
| --- | --- | --- |
| var-1 | Temperature | degC |
| var-2 | Pressure | bar |

Read the latest second from a live source. `start_ms=-1000` and `end_ms=0` use a relative window, so timestamps from an earlier discovery cell cannot expire. Keep the window shorter than the rolling buffer's retention. For stopped recordings, use explicit absolute bounds or `measurements.ipynb`. `points` is an approximate plotting budget, not a raw sample count.

For original cloud samples, replace `points=500` with `resolution=Resolution.RAW` after importing `Resolution` from `gi_data.mapping.enums`. Explicit fetch resolution is cloud-only; do not pass both controls.

In [ ]:
variable_id = "your-variable-id"
variable = {str(v.id): v for v in variables}[variable_id]
selectors = [VarSelector(SID=source.id, VID=variable.id)]

df = client.fetch_buffer(selectors, start_ms=-1000, end_ms=0, points=500)
if df.empty:
    raise RuntimeError("No samples returned; check the variable and time range.")
df = df.rename(columns={str(variable.id): variable.name})
df.head()

**Example output** — local HTTP, after renaming the value column:

| timestamp_ns | Temperature |
| ---: | ---: |
| 1700000000000000000 | 21.5 |
| 1700000000002000000 | 21.7 |

`df.head()` previews at most five rows. On cloud, the index is `time` with UTC datetimes instead.

Time is already the index: UTC datetimes on cloud, backend-origin nanoseconds on local HTTP. Do not call `set_index("time")`.

To save exactly these fetched values, use `df.to_csv("preview.csv", mode="x")`. This is an analysis CSV, not necessarily a GI-native import file.

In [ ]:
import matplotlib.pyplot as plt

ax = df.plot(title=variable.name, legend=False)
ax.set_xlabel("Time (UTC)" if isinstance(df.index, pd.DatetimeIndex) else "Time (ns, backend origin)")
ax.set_ylabel(variable.unit)
plt.tight_layout()
plt.show()
plt.close(ax.figure)

**Example output** — the selected variable plotted against its time index

## Online values

Read up to three current values. This does not use buffer IDs or fetched data.

In [ ]:
online_variables = client.list_variables()[:3]
if not online_variables:
    raise RuntimeError("No readable online variables.")
values = client.read_online([v.id for v in online_variables])
pd.DataFrame([
    {"id": str(v.id), "name": v.name, "unit": v.unit, "value": values[v.id]}
    for v in online_variables
])

**Example output**

| id | name | unit | value |
| --- | --- | --- | ---: |
| var-1 | Temperature | degC | 21.5 |
| var-2 | Pressure | bar | 1.02 |

One current value per variable, not a time series.

## Native file export

**Setup only.** Edit the IDs below to export the latest second of a live buffer. The relative window avoids stale discovery timestamps. Export returns bytes; exclusive creation avoids overwriting files in the notebook's working directory.

In [ ]:
from pathlib import Path

run_export = False
source_id = "your-stream-id"
variable_id = "your-variable-id"

if run_export:
    export_selectors = [VarSelector(SID=source_id, VID=variable_id)]
    payload = client.export_udbf(
        export_selectors, start_ms=-1000, end_ms=0, timezone="UTC",
    )
    if not payload:
        raise RuntimeError("Empty export.")
    export_path = Path("export.dat")
    with export_path.open("xb") as output:
        output.write(payload)
    print(f"{export_path.name}: {len(payload)} bytes")
else:
    print("Export skipped. Set run_export = True to create the file.")

**Example output:**

```text
export.dat: 12345 bytes
```

For native CSV, use `export_csv()` and a `.csv` filename. Inspect separators, decimals, and metadata rows before parsing; raw/local and aggregated cloud layouts differ. Native exports may use a point budget, so verify coverage and sample counts rather than assuming lossless output. For large exports, save one window at a time.

## File import

**Writes target data; use a disposable test stream. Setup only.** Set `input_path` to your own file, not a result from another example. These settings describe a single header row, one timestamp column, semicolons, dot decimals, and timestamps like `2026-01-01 12:00:00.000000` (`%F` is backend syntax).

In [ ]:
from pathlib import Path
from uuid import uuid4
from gi_data.mapping.models import CSVImportSettings

run_import = False
input_path = "your-import.csv"

if run_import:
    payload = Path(input_path).read_bytes()
    if not payload:
        raise ValueError("The import file is empty.")
    settings = CSVImportSettings(
        ColumnSeparator=";", DecimalSeparator=".",
        NameRowIndex=0, UnitRowIndex=-1,
        ValuesStartRowIndex=1, ValuesStartColumnIndex=1,
        DateTimeFmtColumn1="%Y-%m-%d %H:%M:%S.%F",
    )
    new_source_id = str(uuid4())
    session_id = client.import_csv(
        new_source_id, "Tutorial import", payload, csv_settings=settings, target="stream",
    )
    print({"source_id": new_source_id, "import_session_id": session_id})
else:
    print("Import skipped. Set run_import = True to upload the file.")

**Example output:**

```text
{'source_id': '<new stream UUID>', 'import_session_id': '<session ID>'}
```

These are different identifiers; the session ID is not the destination stream ID.

Adapt settings to the actual file. For UDBF, use `import_udbf()` with a fresh source ID and omit CSV settings. Imports are not assumed idempotent; rediscover the destination to inspect the result.

## Online subscription

**Setup only.** Set `online_variable_id` to a known online variable UUID. This does not require the online-read cell. Jupyter supports top-level `await`; scripts use `asyncio.run(show_updates())`.

In [ ]:
import asyncio
from contextlib import aclosing
from uuid import UUID

run_subscription = False
online_variable_id = "your-online-variable-uuid"


async def show_updates():
    updates = client.stream_online(
        [UUID(online_variable_id)], interval_ms=1000, on_change=False,
    )
    async with aclosing(updates):
        for _ in range(3):
            print(await asyncio.wait_for(anext(updates), timeout=10))


if run_subscription:
    await show_updates()
else:
    print("Subscription skipped. Set run_subscription = True to receive updates.")

**Expected output:** three printed server payloads, not a DataFrame; their fields depend on the server. The iterator closes, but public `client.close()` currently closes HTTP only; restart/shut down the kernel to release the WebSocket after this experiment.

## Online write

**Changes an output/setpoint. Setup only.** Use an isolated test target; confirm the output UUID, unit, allowed range, and physical effect before setting `output_variable_id` and `output_value` below.

In [ ]:
from uuid import UUID

run_write = False
output_variable_id = "your-output-variable-uuid"
output_value = None

if run_write:
    if output_value is None:
        raise ValueError("Set an approved output_value before writing.")
    output_id = UUID(output_variable_id)
    client.write_online({output_id: output_value})
    print(client.read_online([output_id]))
else:
    print("Write skipped. Set run_write = True only after approving the output and value.")

**Example output** — the read-back maps the output UUID to its current value:

```text
{UUID('00000000-0000-0000-0000-000000000001'): 1.0}
```

The write itself returns `None`; a read-back is not proof that an attached process is safe.

## Close

Run when finished, including after errors. No displayed output is expected. Re-run Setup before further API calls. In scripts, prefer `with GIDataClient(...) as client:`.

In [ ]:
client.close()